# BTC Barrier Model — Final Experiments

Three questions left, all sharing one `run_experiment()` code path:

1. **Per-family flow ablation** — which MMT family (if any) earned the +1.18pp at h30?
   Time-sensitive: the subscription is cancelled, so this cannot be re-run later.
2. **Asymmetric barrier sweep** — judged by **expected value after costs**, not accuracy.
3. **Walk-forward validation** — every result so far rests on one 80/20 split with
   ~646 effective observations. Multiple folds is what makes 57.3% credible or not.

### Where we stand

| run | features | label | acc | edge vs constant |
|---|---|---|---|---|
| ohlcv close h60 | 50 | close | 51.91% | +0.84pp |
| mmt close h1 | 71 | close | 52.05% | +2.01pp |
| ohlcv barrier h30 | 50 | ±300 / 30m | 53.14% | +2.17pp |
| mmt barrier h30 | 65 | ±300 / 30m | 54.33% | **+3.35pp** |
| **ohlcv barrier h120** | **50** | **±300 / 120m** | **57.30%** | **+4.18pp** |
| mmt barrier h120 | 65 | ±300 / 120m | 52.78% | −0.33pp |

Momentum baseline: 55.6% at h30 (beats both models there), 50.7% at h120 (dead).

**The data is fetched once and reused in memory** — every experiment after that is seconds.

## 0. Config

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

BQ_PROJECT = "trading-brains"
BQ_TABLE   = "trading-brains.market_microstructure.mmt_btc_1m_v3"
LOOKBACK_DAYS = 360

# --- flow families (for the ablation). [] = price only, "all" = every family ---
FLOW_FAMILIES = {
    "vd":      ["vd_retail_ratio", "vd_mid_ratio", "vd_whale_ratio", "vd_whale_vs_retail"],
    "liq":     ["liq_intensity", "liq_imbalance", "liq_vs_volume"],
    "oi":      ["oi_change_1m", "oi_change_15m", "oi_vs_sma"],
    "funding": ["funding_rate", "funding_z"],
    "delta":   ["candle_delta_ratio", "avg_trade_size_ratio"],
    "mark":    ["mark_dislocation_bps"],
}

# --- defaults (each experiment overrides as needed) ---
DEF_UP, DEF_DN, DEF_HOLD = 300.0, 300.0, 120

# --- economics ---
COST_USD = 55.0     # round-trip cost per BTC: ~0.04%/side taker at ~$65k + slippage

# --- model ---
ENC_LEN    = 60
MAX_EPOCHS = 30
BATCH_SIZE = 1024
LR         = 5e-4
D_MODEL, NHEAD, N_LAYERS, DROPOUT, PATIENCE = 64, 4, 2, 0.1, 7

import torch
USE_GPU   = torch.cuda.is_available()
ACCEL     = "gpu" if USE_GPU else "cpu"
PRECISION = "16-mixed" if USE_GPU else "32-true"
print(f"Device: {ACCEL}" + (f" ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

## 1. Imports

In [ ]:
import time, gc, itertools, warnings
from datetime import datetime
import numpy as np, pandas as pd, torch, torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from google.cloud import bigquery

warnings.filterwarnings("ignore")
pl_log = __import__("logging").getLogger("lightning.pytorch")
pl_log.setLevel(__import__("logging").ERROR)      # quiet the per-epoch spam in sweeps
if USE_GPU: torch.set_float32_matmul_precision("high")
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()}")

## 2. Fetch once

Pulls the full 360 days with every full-history flow column. Tape speed and order book
stay out — they are NaN/zero before mid-January and would halve the window.

In [ ]:
def fetch_once(lookback_days=360):
    client = bigquery.Client(project=BQ_PROJECT)
    q = f"""
    WITH bf AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, open, high, low, close, candle_total_vol, candle_delta,
               candle_total_trades, mark_price, funding_rate,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef') WHERE rn = 1),
    agg AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, vd_b2, vd_b3, vd_b4, vd_b5, vd_b6, vd_b7, vd_b8, vd_b9,
               vd_b10, vd_b11, net_liq, liq_total, oi_close,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef:bybitf') WHERE rn = 1)
    SELECT bf.ts AS timestamp, bf.open, bf.high, bf.low, bf.close,
           bf.candle_total_vol AS volume, bf.candle_delta, bf.candle_total_trades,
           bf.funding_rate, bf.mark_price,
           agg.vd_b2, agg.vd_b3, agg.vd_b4, agg.vd_b5, agg.vd_b6, agg.vd_b7,
           agg.vd_b8, agg.vd_b9, agg.vd_b10, agg.vd_b11,
           agg.net_liq, agg.liq_total, agg.oi_close
    FROM bf JOIN agg USING (ts)
    WHERE bf.close > 0
      AND bf.ts >= TIMESTAMP_SUB((SELECT MAX(ts) FROM `{BQ_TABLE}`),
                                 INTERVAL {lookback_days} DAY)
    ORDER BY bf.ts
    """
    print("Querying BigQuery (once)...")
    df = client.query(q).to_dataframe()
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["vd_retail"] = df[["vd_b2","vd_b3"]].sum(axis=1)
    df["vd_mid"]    = df[[f"vd_b{i}" for i in range(4,10)]].sum(axis=1)
    df["vd_whale"]  = df[["vd_b10","vd_b11"]].sum(axis=1)
    df = df.drop(columns=[f"vd_b{i}" for i in range(2,12)])
    print(f"Loaded {len(df):,} bars: {df.timestamp.min()} -> {df.timestamp.max()}")
    return df

RAW = fetch_once(LOOKBACK_DAYS)
RAW.tail(2)

## 3. Features & labels

In [ ]:
def price_features(df):
    df = df.copy()
    for p in [1,5,15,30,60]: df[f"returns_{p}m"] = df["close"].pct_change(p)
    rng = df["high"] - df["low"] + 1e-10
    df["high_low_ratio"]   = df["high"]/df["low"]
    df["high_close_ratio"] = df["high"]/df["close"]
    df["low_close_ratio"]  = df["low"]/df["close"]
    df["upper_shadow"] = (df["high"]-np.maximum(df["open"],df["close"]))/rng
    df["lower_shadow"] = (np.minimum(df["open"],df["close"])-df["low"])/rng
    for p in [5,10,20,30]:
        sma = df["close"].rolling(p).mean()
        df[f"sma_{p}_slope"] = sma.pct_change()
        df[f"close_to_sma_{p}"] = (df["close"]-sma)/sma
    for p in [60,120]:
        sma = df["close"].rolling(p).mean()
        df[f"close_to_sma_{p}"] = (df["close"]-sma)/sma
    df["sma_120_slope"] = df["close"].rolling(120).mean().pct_change()
    e12, e26 = df["close"].ewm(span=12,adjust=False).mean(), df["close"].ewm(span=26,adjust=False).mean()
    macd = e12-e26
    df["macd_norm"] = macd/df["close"]
    df["macd_hist_norm"] = (macd-macd.ewm(span=9,adjust=False).mean())/df["close"]
    for p in [5,10,20,60]: df[f"volatility_{p}"] = df["returns_1m"].rolling(p).std()
    hl = df["high"]-df["low"]
    hc = (df["high"]-df["close"].shift()).abs()
    lc = (df["low"]-df["close"].shift()).abs()
    tr = np.maximum(hl, np.maximum(hc,lc))
    df["atr_60_abs"]  = tr.rolling(60).mean()
    df["atr_14_norm"] = tr.rolling(14).mean()/df["close"]
    df["atr_60_norm"] = df["atr_60_abs"]/df["close"]
    for p in [20,60]:
        sma, std = df["close"].rolling(p).mean(), df["close"].rolling(p).std()
        df[f"bb_position_{p}"] = (df["close"]-sma)/(2*std+1e-10)
        df[f"bb_width_{p}"] = std/sma
    def rsi(s,p):
        d = s.diff(); up = d.where(d>0,0).rolling(p).mean(); dn = (-d.where(d<0,0)).rolling(p).mean()
        return 100-100/(1+up/(dn+1e-10))
    for p in [14,20,60]: df[f"rsi_{p}"] = rsi(df["close"],p)
    for p in [14,60]:
        ll, hh = df["low"].rolling(p).min(), df["high"].rolling(p).max()
        df[f"stoch_k_{p}"] = 100*(df["close"]-ll)/(hh-ll+1e-10)
    df["stoch_d_14"] = df["stoch_k_14"].rolling(3).mean()
    df["volume_change"] = df["volume"].pct_change(1)
    for p in [5,10,20,60]:
        df[f"volume_ratio_{p}"] = df["volume"]/(df["volume"].rolling(p).mean()+1e-10)
    def mfi(d,p):
        tp = (d["high"]+d["low"]+d["close"])/3; mf = tp*d["volume"]
        pos = mf.where(tp>tp.shift(),0).rolling(p).sum()
        neg = mf.where(tp<tp.shift(),0).rolling(p).sum()
        return 100-100/(1+pos/(neg+1e-10))
    df["mfi_14"], df["mfi_60"] = mfi(df,14), mfi(df,60)
    for p in [20,60]:
        vw = (df["volume"]*df["close"]).rolling(p).sum()/(df["volume"].rolling(p).sum()+1e-10)
        df[f"close_to_vwap_{p}"] = (df["close"]-vw)/vw
    obv = (np.sign(df["close"].diff()).fillna(0)*df["volume"]).cumsum()
    df["obv_slope_20"] = obv.diff(20)/(df["volume"].rolling(20).sum()+1e-10)
    h,m,d_ = df.timestamp.dt.hour, df.timestamp.dt.minute, df.timestamp.dt.dayofweek
    df["hour_sin"],df["hour_cos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
    df["minute_sin"],df["minute_cos"] = np.sin(2*np.pi*m/60), np.cos(2*np.pi*m/60)
    df["day_sin"],df["day_cos"] = np.sin(2*np.pi*d_/7), np.cos(2*np.pi*d_/7)
    return df


def flow_features(df):
    """Builds ALL flow families; selection happens later by column name."""
    df = df.copy(); vol = df["volume"]+1e-10
    for c in ["vd_retail","vd_mid","vd_whale"]: df[f"{c}_ratio"] = df[c]/vol
    df["vd_whale_vs_retail"] = df["vd_whale_ratio"]-df["vd_retail_ratio"]
    df["candle_delta_ratio"] = df["candle_delta"]/vol
    ats = df["volume"]/(df["candle_total_trades"]+1e-10)
    df["avg_trade_size_ratio"] = ats/(ats.rolling(60).mean()+1e-10)
    df["liq_intensity"] = df["liq_total"]/(df["liq_total"].rolling(240).mean()+1e-10)
    df["liq_imbalance"] = df["net_liq"]/(df["liq_total"]+1e-10)
    df["liq_vs_volume"] = df["liq_total"]/vol
    df["oi_change_1m"]  = df["oi_close"].pct_change(1)
    df["oi_change_15m"] = df["oi_close"].pct_change(15)
    oi_sma = df["oi_close"].rolling(240).mean()
    df["oi_vs_sma"] = (df["oi_close"]-oi_sma)/(oi_sma+1e-10)
    fr = df["funding_rate"]
    df["funding_z"] = (fr-fr.rolling(480).mean())/(fr.rolling(480).std()+1e-10)
    df["mark_dislocation_bps"] = (df["close"]-df["mark_price"])/df["close"]*1e4
    return df.drop(columns=["vd_retail","vd_mid","vd_whale","candle_delta",
                            "candle_total_trades","net_liq","liq_total",
                            "oi_close","mark_price"])


def barrier_labels(df, up_usd, dn_usd, hold, atr_mult=None):
    """First touch of +up_usd vs -dn_usd within `hold` bars. 1=up first, 0=down first."""
    close, high, low = df["close"].to_numpy(), df["high"].to_numpy(), df["low"].to_numpy()
    n = len(df)
    if atr_mult is not None:
        a = df["atr_60_abs"].to_numpy()
        ups, dns = a*atr_mult, a*atr_mult
    else:
        ups = np.full(n, float(up_usd)); dns = np.full(n, float(dn_usd))
    out = np.full(n, np.nan, dtype="float32")
    for i in range(n-1):
        u, d = ups[i], dns[i]
        if not (np.isfinite(u) and np.isfinite(d)) or u <= 0 or d <= 0: continue
        hi_t, lo_t = close[i]+u, close[i]-d
        end = min(i+hold, n-1)
        for j in range(i+1, end+1):
            hu, hd = high[j] >= hi_t, low[j] <= lo_t
            if hu and hd: break                 # ambiguous bar — discard
            if hu: out[i] = 1.0; break
            if hd: out[i] = 0.0; break
    return out

## 4. `run_experiment()` — one code path for everything

In [ ]:
DEV = torch.device("cuda" if USE_GPU else "cpu")

class Bank:
    def __init__(self, X, y, enc):
        self.X = torch.from_numpy(X).to(DEV); self.y = torch.from_numpy(y).to(DEV)
        self.enc = enc; self.n = len(X)-enc+1
        self.off = torch.arange(enc, device=DEV)
    def gather(self, idx):
        return self.X[idx.unsqueeze(1)+self.off.unsqueeze(0)], self.y[idx+self.enc-1]

class IdxDS(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i): return i

class Clf(pl.LightningModule):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2,
                 dropout=0.1, learning_rate=5e-4, focal_alpha=0.5, focal_gamma=1.5):
        super().__init__(); self.save_hyperparameters()
        self.bank_train = self.bank_val = None
        self.inp = nn.Linear(n_features, d_model)
        self.pos = nn.Parameter(torch.randn(1,256,d_model)*0.02)
        lay = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                dim_feedforward=d_model*4, dropout=dropout,
                batch_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(lay, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model,d_model//2),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model//2,1))
    def forward(self,x):
        h = self.inp(x); h = h + self.pos[:,:h.size(1),:]
        return self.head(self.tr(h)[:,-1,:]).squeeze(-1)
    def _loss(self,lg,y):
        bce = nn.functional.binary_cross_entropy_with_logits(lg,y,reduction="none")
        p = torch.sigmoid(lg); pt = p*y+(1-p)*(1-y)
        at = self.hparams.focal_alpha*y+(1-self.hparams.focal_alpha)*(1-y)
        return (at*(1-pt)**self.hparams.focal_gamma*bce).mean()
    def _step(self,idx,bank,stage):
        x,y = bank.gather(idx); lg = self(x); loss = self._loss(lg,y)
        self.log(f"{stage}_loss", loss, batch_size=len(idx))
        return loss
    def training_step(self,i,_):   return self._step(i,self.bank_train,"train")
    def validation_step(self,i,_): return self._step(i,self.bank_val,"val")
    def configure_optimizers(self):
        o = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)
        return [o],[torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=self.trainer.max_epochs)]


def build_frame(families, up, dn, hold, atr_mult=None):
    """families: list of family names, or [] for price-only."""
    df = price_features(RAW)
    if families: df = flow_features(df)
    df["target"] = barrier_labels(df, up, dn, hold, atr_mult)

    keep_flow = [c for f in families for c in FLOW_FAMILIES[f]]
    drop_flow = [c for f, cols in FLOW_FAMILIES.items() for c in cols
                 if c not in keep_flow and c in df.columns]
    df = df.drop(columns=drop_flow, errors="ignore")

    df = df.replace([np.inf,-np.inf], np.nan)
    fc = [c for c in df.columns if c not in ("timestamp","target")]
    df = df.dropna(subset=fc).dropna(subset=["target"]).reset_index(drop=True)

    excl = {"timestamp","target","open","high","low","close","volume","atr_60_abs"}
    feats = [c for c in df.columns if c not in excl and pd.api.types.is_numeric_dtype(df[c])]
    corr = df[feats].corr().abs()
    up_tri = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
    drop = [c for c in up_tri.columns if any(up_tri[c] > 0.95) and c in keep_flow] + \
           [c for c in up_tri.columns if any(up_tri[c] > 0.95) and c not in keep_flow]
    drop = list(dict.fromkeys(drop))
    feats = [c for c in feats if c not in drop]
    return df, feats


def ev_per_trade(probs, labels, up, dn, thresh=0.5, cost=None):
    """Dollars per BTC per trade, after costs, trading only |p-0.5| >= thresh-0.5."""
    cost = COST_USD if cost is None else cost
    take = np.abs(probs-0.5) >= (thresh-0.5)
    if take.sum() == 0: return dict(n=0, ev=np.nan, acc=np.nan, total=0.0)
    p, l = probs[take], labels[take]
    long_ = p > 0.5
    pnl = np.where(long_, np.where(l==1,  up, -dn),
                          np.where(l==0,  dn, -up)) - cost
    acc = np.where(long_, l==1, l==0).mean()
    return dict(n=int(take.sum()), ev=float(pnl.mean()), acc=float(acc), total=float(pnl.sum()))


def run_experiment(families=(), up=DEF_UP, dn=DEF_DN, hold=DEF_HOLD, atr_mult=None,
                   folds=1, tag="", verbose=True, max_epochs=MAX_EPOCHS):
    """folds=1 -> single 80/20 split. folds>1 -> expanding-window walk-forward."""
    df, feats = build_frame(list(families), up, dn, hold, atr_mult)
    name = tag or f"{'+'.join(families) if families else 'price'}_{int(up)}/{int(dn)}_h{hold}"
    fold_out = []

    for k in range(folds):
        if folds == 1:
            tr_end = int(len(df)*0.8); va_end = len(df)
        else:                                  # expanding window
            frac0 = 0.5 + k*(0.5/folds)
            tr_end = int(len(df)*frac0)
            va_end = int(len(df)*(frac0 + 0.5/folds))
        dtr, dva = df.iloc[:tr_end], df.iloc[tr_end:va_end]
        if len(dva) < ENC_LEN*3: continue

        sc = StandardScaler().fit(dtr[feats].values)
        btr = Bank(sc.transform(dtr[feats].values).astype("float32"),
                   dtr["target"].values.astype("float32"), ENC_LEN)
        bva = Bank(sc.transform(dva[feats].values).astype("float32"),
                   dva["target"].values.astype("float32"), ENC_LEN)

        m = Clf(len(feats), D_MODEL, NHEAD, N_LAYERS, DROPOUT, LR)
        m.bank_train, m.bank_val = btr, bva
        coll = lambda b: torch.tensor(b, device=DEV)
        tdl = DataLoader(IdxDS(btr.n), batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=coll, drop_last=True)
        vdl = DataLoader(IdxDS(bva.n), batch_size=BATCH_SIZE, shuffle=False, collate_fn=coll)
        ck = ModelCheckpoint(dirpath=f"ckpt_tmp/{abs(hash(name))}_{k}", monitor="val_loss",
                             save_top_k=1, mode="min")
        tr_ = pl.Trainer(max_epochs=max_epochs, accelerator=ACCEL, devices=1,
                         precision=PRECISION, gradient_clip_val=1.0,
                         callbacks=[EarlyStopping("val_loss", patience=PATIENCE, mode="min"), ck],
                         enable_progress_bar=False, enable_model_summary=False, logger=False)
        tr_.fit(m, tdl, vdl)

        best = Clf.load_from_checkpoint(ck.best_model_path, n_features=len(feats))
        best.eval().to(DEV)
        P, L = [], []
        with torch.no_grad():
            for idx in vdl:
                x,y = bva.gather(idx)
                P.append(torch.sigmoid(best(x)).float().cpu()); L.append(y.float().cpu())
        pr, lb = torch.cat(P).numpy(), torch.cat(L).numpy()

        base = lb.mean(); const = max(base, 1-base)
        acc = ((pr>0.5)==lb).mean()
        n_eff = max(1, len(lb)//hold)
        fold_out.append(dict(
            fold=k, n=len(lb), n_eff=n_eff, base=float(base), acc=float(acc),
            edge_pp=float((acc-const)*100),
            sigma=float(abs(acc-const)*100/(np.sqrt(0.25/n_eff)*100)),
            brier=float(np.mean((pr-lb)**2)),
            brier_const=float(np.mean((base-lb)**2)),
            ev50=ev_per_trade(pr,lb,up,dn,0.50)["ev"],
            ev60=ev_per_trade(pr,lb,up,dn,0.60)["ev"],
            n60=ev_per_trade(pr,lb,up,dn,0.60)["n"],
            ev70=ev_per_trade(pr,lb,up,dn,0.70)["ev"],
            n70=ev_per_trade(pr,lb,up,dn,0.70)["n"],
            probs=pr, labels=lb))

    if not fold_out: return None
    agg = dict(name=name, families=list(families), up=up, dn=dn, hold=hold,
               n_features=len(feats), folds=len(fold_out),
               acc=float(np.mean([f["acc"] for f in fold_out])),
               edge_pp=float(np.mean([f["edge_pp"] for f in fold_out])),
               edge_min=float(np.min([f["edge_pp"] for f in fold_out])),
               brier=float(np.mean([f["brier"] for f in fold_out])),
               brier_const=float(np.mean([f["brier_const"] for f in fold_out])),
               ev50=float(np.nanmean([f["ev50"] for f in fold_out])),
               ev60=float(np.nanmean([f["ev60"] for f in fold_out])),
               ev70=float(np.nanmean([f["ev70"] for f in fold_out])),
               n70=int(np.sum([f["n70"] for f in fold_out])),
               _folds=fold_out)
    if verbose:
        print(f"{name:<34} feat={agg['n_features']:<3} acc={agg['acc']*100:5.2f}% "
              f"edge={agg['edge_pp']:+5.2f}pp (worst fold {agg['edge_min']:+5.2f}) "
              f"brier={agg['brier']:.4f}{'<' if agg['brier']<agg['brier_const'] else '>'}const "
              f"EV50=${agg['ev50']:6.1f} EV70=${agg['ev70']:6.1f} (n={agg['n70']})")
    return agg

print("run_experiment() ready.")

## 5. Per-family flow ablation  ← run this first

MMT access is lapsing. This is the only experiment that cannot be repeated later.

Run at **h30**, where flow demonstrably helped (+1.18pp), and at **h120**, where the
blended flow block *hurt* (−4.51pp). If one family carries the h30 benefit, a lean
version might help at h120 too, rather than 15 columns of noise swamping the signal.

In [ ]:
ABLATION = []
for hold in [30, 120]:
    print(f"\n--- hold {hold}m, +/-$300 ---")
    ABLATION.append(run_experiment(families=(), hold=hold, tag=f"price_only_h{hold}"))
    for fam in FLOW_FAMILIES:
        ABLATION.append(run_experiment(families=(fam,), hold=hold, tag=f"{fam}_only_h{hold}"))
    ABLATION.append(run_experiment(families=tuple(FLOW_FAMILIES), hold=hold,
                                   tag=f"all_flow_h{hold}"))

In [ ]:
# Ablation summary: each family's lift over price-only, per hold
for hold in [30, 120]:
    rows = [r for r in ABLATION if r and r["hold"] == hold]
    base = next(r for r in rows if r["name"].startswith("price_only"))
    print(f"\nHOLD {hold}m — lift over price-only (edge {base['edge_pp']:+.2f}pp, "
          f"EV50 ${base['ev50']:.1f})")
    print(f"  {'family':<16}{'edge pp':>9}{'lift':>8}{'EV50':>9}{'EV lift':>9}{'feat':>6}")
    for r in rows:
        if r is base: continue
        fam = r["name"].split("_")[0]
        print(f"  {fam:<16}{r['edge_pp']:>+9.2f}{r['edge_pp']-base['edge_pp']:>+8.2f}"
              f"{r['ev50']:>9.1f}{r['ev50']-base['ev50']:>+9.1f}{r['n_features']:>6}")
    print("  A family earns its keep only if lift is positive at BOTH holds.")

## 6. Asymmetric barrier sweep

**Judged by EV after costs, not accuracy** — they rank differently. A 40% hit rate on
+$400/−$200 beats 55% on ±$300.

Read it the way the XV grid was finally read: look for a **plateau**, not a peak.
A single bright cell surrounded by flat ones is overfitting; a broad profitable
region is a real, parameter-insensitive edge.

In [ ]:
BEST_FAMILIES = ()        # <- set from section 5 (e.g. ("liq",)); () = price only

UP_GRID   = [200, 300, 400, 600]
DN_GRID   = [150, 200, 300, 400]
HOLD_GRID = [60, 120]

SWEEP = []
for hold in HOLD_GRID:
    for up in UP_GRID:
        for dn in DN_GRID:
            r = run_experiment(families=BEST_FAMILIES, up=up, dn=dn, hold=hold,
                               tag=f"{'+'.join(BEST_FAMILIES) or 'price'}_{up}/{dn}_h{hold}")
            if r: SWEEP.append(r)
print(f"\n{len(SWEEP)} cells done.")

In [ ]:
# Plateau map: EV per trade at p>0.5. "." = negative EV.
for hold in HOLD_GRID:
    print(f"\nEV per trade ($, after ${COST_USD:.0f} costs) — hold {hold}m")
    print(f"  {'TP\\SL':<8}" + "".join(f"{dn:>8}" for dn in DN_GRID))
    for up in UP_GRID:
        row = ""
        for dn in DN_GRID:
            c = next((s for s in SWEEP if s["up"]==up and s["dn"]==dn and s["hold"]==hold), None)
            row += f"{c['ev50']:>8.0f}" if c and np.isfinite(c["ev50"]) else f"{'-':>8}"
        print(f"  {up:<8}" + row)

best = sorted([s for s in SWEEP if np.isfinite(s["ev50"])],
              key=lambda s: -s["ev50"])[:8]
print(f"\nTop cells by EV (treat as candidates, not winners):")
print(f"  {'cell':<26}{'acc':>7}{'edge':>8}{'EV50':>8}{'EV70':>8}{'n70':>7}")
for s in best:
    print(f"  {s['name']:<26}{s['acc']*100:>6.1f}%{s['edge_pp']:>+8.2f}"
          f"{s['ev50']:>8.0f}{s['ev70']:>8.0f}{s['n70']:>7,}")

## 7. Walk-forward validation  ← the one that decides credibility

Every result so far is a single 80/20 split with ~646 effective observations.
Expanding-window folds test whether the edge survives being asked repeatedly,
across different periods.

**The bar: positive in every fold.** One negative fold means the result is regime
luck — the same signature that killed every XV configuration.

In [ ]:
WF_CONFIGS = [
    dict(families=(),            up=300, dn=300, hold=120, tag="price_300/300_h120"),
    dict(families=(),            up=300, dn=300, hold=30,  tag="price_300/300_h30"),
    # add the best sweep cell here once section 6 has run
]

WF = []
for cfg in WF_CONFIGS:
    r = run_experiment(folds=5, verbose=False, **cfg)
    if not r: continue
    WF.append(r)
    print(f"\n{r['name']}  ({r['folds']} folds)")
    print(f"  {'fold':<6}{'n':>9}{'base':>8}{'acc':>8}{'edge':>9}{'EV50':>9}")
    for f in r["_folds"]:
        print(f"  {f['fold']:<6}{f['n']:>9,}{f['base']*100:>7.1f}%{f['acc']*100:>7.1f}%"
              f"{f['edge_pp']:>+9.2f}{f['ev50']:>9.0f}")
    pos = sum(1 for f in r["_folds"] if f["edge_pp"] > 0)
    print(f"  MEAN edge {r['edge_pp']:+.2f}pp | worst fold {r['edge_min']:+.2f}pp "
          f"| positive in {pos}/{r['folds']} folds "
          f"-> {'PASSES' if pos == r['folds'] else 'FAILS'} the all-folds bar")

## 8. Entry frequency — does minute-by-minute beat hourly?

In [ ]:
# Uses the first walk-forward config's last fold.
if WF:
    f = WF[0]["_folds"][-1]
    pr, lb = f["probs"], f["labels"]
    hold = WF[0]["hold"]; up = WF[0]["up"]; dn = WF[0]["dn"]
    print(f"{WF[0]['name']} — sampling every Nth minute\n")
    print(f"  {'every':<10}{'n':>9}{'acc':>8}{'EV50':>9}{'EV70':>9}{'n70':>7}")
    for step in [1, 5, 15, 30, 60]:
        sel = np.arange(0, len(pr), step)
        e50 = ev_per_trade(pr[sel], lb[sel], up, dn, 0.50)
        e70 = ev_per_trade(pr[sel], lb[sel], up, dn, 0.70)
        acc = ((pr[sel] > 0.5) == lb[sel]).mean()
        print(f"  {step:<10}{len(sel):>9,}{acc*100:>7.1f}%{e50['ev']:>9.0f}"
              f"{e70['ev']:>9.0f}{e70['n']:>7,}")
    print("\n  Accuracy should barely move (overlapping windows give near-identical")
    print("  predictions). If it does move, entry timing matters more than expected.")
else:
    print("Run section 7 first.")